# WindGenerator — Demo Notebook

End-to-end wind audio generation using a DDPM diffusion model trained on mel spectrograms.

**Pipeline:** Gaussian noise → DDPM denoising → mel spectrogram → Griffin-Lim → audio

**Runtime:** This notebook requires a GPU. Go to `Runtime → Change runtime type → T4 GPU`.

**Pretrained model:** You need a trained checkpoint. Either:
- Train from scratch using `scripts/train_diffusion.py`
- Use a checkpoint saved to Google Drive from a previous training session

## 1. Setup

In [ ]:
# Clone repo and install
!git clone https://github.com/alpercagan/WindGenerator.git /content/WindGenerator 2>/dev/null || \
    (cd /content/WindGenerator && git pull)
!pip install -e /content/WindGenerator -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Mount Google Drive (needed to load checkpoint and mel_stats)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set paths — edit these to match your Drive structure
CHECKPOINT_PATH = '/content/drive/MyDrive/WindGenerator/ckpt_step_074000.pt'
MEL_STATS_PATH  = '/content/drive/MyDrive/WindGenerator/mel_stats.json'
OUTPUT_DIR      = '/content/generated'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify files exist
for path, name in [(CHECKPOINT_PATH, 'Checkpoint'), (MEL_STATS_PATH, 'Mel stats')]:
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1024**2 if exists else 0
    print(f"{'✅' if exists else '❌'} {name}: {path} ({size:.1f} MB)")

## 2. Load Model

The diffusion model is a `UNet2DModel` with ~2.5M parameters, trained to denoise mel spectrograms. Architecture is inferred automatically from the checkpoint.

In [ ]:
import torch, json, sys
import numpy as np
sys.path.insert(0, '/content/WindGenerator/src')

from diffusers import UNet2DModel, DDPMScheduler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
state_dict = ckpt.get('model', ckpt)

# Infer architecture from checkpoint weights
c0 = state_dict['conv_in.weight'].shape[0]
block_out_channels = (c0, c0*2, c0*4)
has_resnet1 = any('resnets.1' in k for k in state_dict.keys() 
                   if 'down_blocks.0' in k)
layers_per_block = 2 if has_resnet1 else 1
n_levels = len([k for k in state_dict.keys() 
                if k.startswith('down_blocks.') and 
                k.endswith('resnets.0.norm1.weight')])

down_types = tuple(['DownBlock2D'] * n_levels)
up_types   = tuple(['UpBlock2D']   * n_levels)

model = UNet2DModel(
    sample_size=(128, 440),
    in_channels=1,
    out_channels=1,
    block_out_channels=block_out_channels,
    layers_per_block=layers_per_block,
    down_block_types=down_types,
    up_block_types=up_types,
).to(device)

model.load_state_dict(state_dict)
model.disable_gradient_checkpointing()
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params:,} parameters")
print(f"Architecture: channels={block_out_channels}, layers_per_block={layers_per_block}")
print(f"Training step: {ckpt.get('step', 'unknown')}")

# Load mel stats for denormalization
with open(MEL_STATS_PATH) as f:
    stats = json.load(f)
mel_mean = stats['mean']
mel_std  = stats['std']
print(f"Mel stats: mean={mel_mean:.4f}, std={mel_std:.4f}")

# Scheduler
scheduler = DDPMScheduler(num_train_timesteps=1000)
print("Ready.")

## 3. Generate Audio

The generation process:
1. Sample Gaussian noise `(1, 128, 440)` — a random "image"
2. Run DDPM denoising for N steps — the model iteratively removes noise, moving toward the learned distribution of wind spectrograms
3. Denormalize the output mel spectrogram
4. Convert mel → linear spectrogram via `InverseMelScale`
5. Reconstruct waveform via Griffin-Lim phase estimation

In [ ]:
import torchaudio
import soundfile as sf
import IPython.display as ipd
from tqdm.auto import tqdm

def generate_clip(ddpm_steps=100, seed=None):
    """Generate one ~10s wind audio clip."""
    if seed is not None:
        torch.manual_seed(seed)

    # Step 1: Sample noise
    x = torch.randn(1, 1, 128, 440, device=device)

    # Step 2: DDPM denoising
    scheduler.set_timesteps(ddpm_steps)
    with torch.no_grad():
        for t in tqdm(scheduler.timesteps, desc='Denoising', leave=False):
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                noise_pred = model(x, t).sample
            x = scheduler.step(noise_pred, t, x).prev_sample

    mel = x.squeeze(0).squeeze(0).cpu()  # (128, 440)
    mel = mel.clamp(-1, 1)

    # Step 3: Denormalize
    # Reverse: scale [-1,1] → z-score → exp → linear power spectrogram
    mel_z = mel * (4.0 * mel_std) + mel_mean   # z-score
    mel_linear = torch.exp(mel_z) - 1e-5        # linear power
    mel_linear = mel_linear.clamp(min=0)
    mel_linear = mel_linear.unsqueeze(0)         # (1, 128, 440)

    # Step 4: InverseMelScale
    inverse_mel = torchaudio.transforms.InverseMelScale(
        n_stft=513, n_mels=128, sample_rate=22050,
        f_min=20.0, f_max=11025.0,
    )
    linear_spec = inverse_mel(mel_linear)  # (1, 513, 440)

    # Step 5: Griffin-Lim
    griffin_lim = torchaudio.transforms.GriffinLim(
        n_fft=1024, hop_length=256, win_length=1024,
        power=2.0, n_iter=64,
    )
    segment = griffin_lim(linear_spec).squeeze(0)  # (T,)

    # Crossfade segment with itself for ~10s output
    fade_len = 11025  # 0.5s
    fade_out = torch.linspace(1, 0, fade_len)
    fade_in  = torch.linspace(0, 1, fade_len)
    seg1 = segment.clone()
    seg2 = segment.clone()
    seg1[-fade_len:] *= fade_out
    seg2[:fade_len]  *= fade_in
    overlap = seg1[-fade_len:] + seg2[:fade_len]
    audio = torch.cat([seg1[:-fade_len], overlap, seg2[fade_len:]])

    # Peak normalize
    peak = audio.abs().max()
    if peak > 0:
        audio = audio / peak * 0.95

    return audio.numpy()

print("Generation function ready.")
print("Run the next cell to generate audio.")

In [ ]:
# Generate and listen — adjust NUM_CLIPS and DDPM_STEPS as needed
NUM_CLIPS  = 5    # number of clips to generate
DDPM_STEPS = 100  # denoising steps (more = better quality, slower)

for i in range(NUM_CLIPS):
    print(f"\n--- Clip {i+1}/{NUM_CLIPS} ---")
    audio = generate_clip(ddpm_steps=DDPM_STEPS, seed=i)
    path = f'{OUTPUT_DIR}/wind_{i+1:02d}.wav'
    sf.write(path, audio, samplerate=22050)
    print(f"Duration: {len(audio)/22050:.1f}s  |  Saved: {path}")
    display(ipd.Audio(path))

## 4. Visualize Generated Spectrograms

Comparing a real wind spectrogram from the dataset to a generated one — this shows what the model has learned to produce.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Generate a mel spectrogram (without converting to audio)
torch.manual_seed(42)
x = torch.randn(1, 1, 128, 440, device=device)
scheduler.set_timesteps(100)

with torch.no_grad():
    for t in tqdm(scheduler.timesteps, desc='Generating spectrogram', leave=False):
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            noise_pred = model(x, t).sample
        x = scheduler.step(noise_pred, t, x).prev_sample

generated_mel = x.squeeze().cpu().numpy()  # (128, 440)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Generated spectrogram
im0 = axes[0].imshow(generated_mel, aspect='auto', origin='lower',
                      cmap='magma', vmin=-1, vmax=1)
axes[0].set_title('Generated mel spectrogram (diffusion model output)', fontsize=11)
axes[0].set_xlabel('Time frames')
axes[0].set_ylabel('Mel frequency bins')
plt.colorbar(im0, ax=axes[0], label='Normalized value')

# Load a real spectrogram for comparison
try:
    import glob
    sys.path.insert(0, '/content/WindGenerator/src')
    from windgen.mels import MelSpecConfig, LogMelExtractor
    
    wav_files = sorted(glob.glob('/content/drive/MyDrive/WindGenerator/clips/*.wav'))
    if wav_files:
        cfg = MelSpecConfig()
        extractor = LogMelExtractor(cfg, stats_path=MEL_STATS_PATH)
        wav, sr = torchaudio.load(wav_files[0])
        if sr != cfg.sr:
            wav = torchaudio.functional.resample(wav, sr, cfg.sr)
        real_mel = extractor(wav).squeeze(0).numpy()[:, :440]  # (128, 440)
        
        im1 = axes[1].imshow(real_mel, aspect='auto', origin='lower',
                              cmap='magma', vmin=-1, vmax=1)
        axes[1].set_title('Real wind recording (from dataset)', fontsize=11)
        axes[1].set_xlabel('Time frames')
        axes[1].set_ylabel('Mel frequency bins')
        plt.colorbar(im1, ax=axes[1], label='Normalized value')
    else:
        axes[1].text(0.5, 0.5, 'Dataset clips not available\n(mount Drive and set path)',
                    ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title('Real wind recording (not loaded)', fontsize=11)
except Exception as e:
    axes[1].text(0.5, 0.5, f'Could not load real mel:\n{e}',
                ha='center', va='center', transform=axes[1].transAxes, fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/spectrogram_comparison.png', bbox_inches='tight')
plt.show()
print("Saved: spectrogram_comparison.png")

## 5. Download Generated Clips

In [ ]:
from google.colab import files
import glob

wav_files = sorted(glob.glob(f'{OUTPUT_DIR}/*.wav'))
print(f"Generated {len(wav_files)} clips:")
for f in wav_files:
    size = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f)} ({size:.0f} KB)")

print("\nDownloading...")
for f in wav_files:
    files.download(f)

---

## Notes

**On output quality:** Some clips will sound more wind-like than others. This reflects the stochastic nature of diffusion sampling — each run starts from a different noise tensor and takes a different path through the learned distribution. The model was trained for 74,000 steps (target: 100,000), so some outputs are better than others.

**On the metallic quality:** The artifact is from Griffin-Lim phase reconstruction, not from the diffusion model. Griffin-Lim estimates phase iteratively but converges to a phase solution that minimizes reconstruction error in aggregate, which sounds slightly metallic. The spectrogram content (frequency distribution, temporal texture) is learned by the diffusion model.

**On varying `ddpm_steps`:** More steps generally produce cleaner outputs but take longer. 50 steps is fast; 100-200 is better quality. Above 200 there is diminishing returns.

**Repository:** [github.com/alpercagan/WindGenerator](https://github.com/alpercagan/WindGenerator)